In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Callable, Dict, Tuple

import numpy as np
import jax
import jax.numpy as jnp
import optax

jax.config.update("jax_enable_x64", True)

DTYPE = jnp.float64
Array = jnp.ndarray


# =============================================================================
# 1) SDE simulation (data only)
# =============================================================================

@dataclass(frozen=True)
class SDEConfig:
    T: float = 2.0
    dt: float = 0.01
    n_traj: int = 512
    x0_low: float = -0.5
    x0_high: float = 0.5

    # simulation-only (unknown in real use)
    sigma0: float = 1.0


def simulate_traj_and_increments(key: Array, cfg: SDEConfig) -> Tuple[Array, Array, Array]:
    """Euler–Maruyama simulation for the demo SDE: dX = sigma0 dW.

    Returns
    -------
    TX_all : (n_traj*(N+1), 2)  rows are [t, x]
    Z      : (n_traj*N, 2)      conditioning states [t_n, x_n]
    dX     : (n_traj*N, 1)      increments X_{n+1} - X_n
    """
    dt = float(cfg.dt)
    N = int(cfg.T / dt)
    t_grid = jnp.linspace(0.0, cfg.T, N + 1, dtype=DTYPE)

    k_init, k_noise = jax.random.split(key, 2)
    x0 = jax.random.uniform(
        k_init, (cfg.n_traj,), minval=cfg.x0_low, maxval=cfg.x0_high, dtype=DTYPE
    )
    dW = jax.random.normal(k_noise, (cfg.n_traj, N), dtype=DTYPE) * math.sqrt(dt)

    def step(x, dWn):
        return x + DTYPE(cfg.sigma0) * dWn, x + DTYPE(cfg.sigma0) * dWn

    _, xs = jax.lax.scan(step, x0, dW.T)
    xs = jnp.concatenate([x0[None, :], xs], axis=0)  # (N+1, n_traj)

    TT = jnp.broadcast_to(t_grid[:, None], xs.shape)
    TX_all = jnp.stack([TT, xs], axis=-1).reshape(-1, 2)

    x_n = xs[:-1, :]
    x_np1 = xs[1:, :]
    t_n = jnp.broadcast_to(t_grid[:-1, None], x_n.shape)

    Z = jnp.stack([t_n, x_n], axis=-1).reshape(-1, 2)
    dX = (x_np1 - x_n).reshape(-1, 1)
    return TX_all, Z, dX


# =============================================================================
# 2) Learn drift/diffusion surrogates from increments
# =============================================================================

@dataclass(frozen=True)
class SurrogateNetConfig:
    hidden: int = 128
    depth: int = 3
    activation: str = "tanh"  # tanh|relu|gelu

    # subtle bias: make affine drifts easy (and derivatives correct)
    use_linear_skip: bool = True
    lin_init_scale: float = 1e-2

    sigma_min: float = 1e-6

    # subtle bias: prefer (nearly) constant diffusion while still allowing general σ(t,x)
    use_amp_residual_sigma: bool = True
    res_scale: float = 1.0
    amp_init: float = 0.05


@dataclass(frozen=True)
class SurrogateTrainConfig:
    steps: int = 6000
    batch_size: int = 8192
    lr: float = 3e-3
    weight_decay: float = 1e-6
    clip_norm: float = 1.0
    print_every: int = 250
    seed: int = 0

    # regularizers (keep small; they bias toward smooth/simple coefficients)
    w_grad_mu: float = 5e-5
    w_grad_sigma: float = 5e-5
    w_amp_l1: float = 1e-3


def _act(x: Array, name: str) -> Array:
    if name == "tanh":
        return jnp.tanh(x)
    if name == "relu":
        return jax.nn.relu(x)
    if name == "gelu":
        return jax.nn.gelu(x)
    raise ValueError(f"unknown activation: {name}")


def _mlp_init(key, in_dim: int, out_dim: int, hidden: int, depth: int) -> Dict[str, Array]:
    keys = jax.random.split(key, depth + 1)
    dims = [in_dim] + [hidden] * depth + [out_dim]
    params = {"W": [], "b": []}
    for i in range(len(dims) - 1):
        fan_in, fan_out = dims[i], dims[i + 1]
        lim = math.sqrt(6.0 / float(fan_in + fan_out))
        W = jax.random.uniform(keys[i], (fan_in, fan_out), minval=-lim, maxval=lim, dtype=DTYPE)
        b = jnp.zeros((fan_out,), dtype=DTYPE)
        params["W"].append(W)
        params["b"].append(b)
    return params


def _mlp_apply(params: Dict[str, Array], x: Array, activation: str) -> Array:
    h = x
    for i in range(len(params["W"]) - 1):
        h = h @ params["W"][i] + params["b"][i]
        h = _act(h, activation)
    return h @ params["W"][-1] + params["b"][-1]


def _normalize_z(Z: Array) -> Tuple[Array, Dict[str, Array]]:
    """Normalize (t,x) to roughly [-1,1] with data-driven stats."""
    mean = jnp.mean(Z, axis=0)
    std = jnp.std(Z, axis=0) + DTYPE(1e-8)
    Zs = (Z - mean) / std
    return Zs, {"mean": mean, "std": std}


def _denorm_z(zs: Array, stats: Dict[str, Array]) -> Array:
    return zs * stats["std"] + stats["mean"]


def init_surrogate_params(key: Array, net: SurrogateNetConfig) -> Dict[str, Dict]:
    k_mu, k_sig, k_lin, k_base, k_amp = jax.random.split(key, 5)
    params = {
        "mu_mlp": _mlp_init(k_mu, 2, 1, net.hidden, net.depth),
        "sig_mlp": _mlp_init(k_sig, 2, 1, net.hidden, net.depth),
    }
    if net.use_linear_skip:
        W = net.lin_init_scale * jax.random.normal(k_lin, (2, 1), dtype=DTYPE)
        b = jnp.zeros((1,), dtype=DTYPE)
        params["mu_lin"] = {"W": W, "b": b}
    else:
        params["mu_lin"] = None

    if net.use_amp_residual_sigma:
        params["sig_base"] = jnp.asarray([1.0], dtype=DTYPE) + 0.01 * jax.random.normal(k_base, (1,), dtype=DTYPE)
        params["sig_amp"] = jnp.asarray([net.amp_init], dtype=DTYPE) + 0.01 * jax.random.normal(k_amp, (1,), dtype=DTYPE)
    else:
        params["sig_base"] = None
        params["sig_amp"] = None

    return params


def surrogate_mu_sigma(
    params: Dict[str, Dict],
    z_norm: Array,
    net: SurrogateNetConfig,
) -> Tuple[Array, Array]:
    """Return (mu, sigma) at normalized inputs z_norm (shape (...,2))."""
    mu = _mlp_apply(params["mu_mlp"], z_norm, net.activation)
    if net.use_linear_skip and params["mu_lin"] is not None:
        mu = mu + (z_norm @ params["mu_lin"]["W"] + params["mu_lin"]["b"])

    sig_raw = _mlp_apply(params["sig_mlp"], z_norm, net.activation)  # residual
    if net.use_amp_residual_sigma and (params["sig_base"] is not None) and (params["sig_amp"] is not None):
        # global amplitude gate -> discourages *derivative leakage* when collapsing to constant
        sig = params["sig_base"] + params["sig_amp"] * jnp.tanh(sig_raw / DTYPE(net.res_scale)) * DTYPE(net.res_scale)
    else:
        sig = sig_raw

    # enforce positivity (and avoid degenerate sigma)
    sig = jax.nn.softplus(sig) + DTYPE(net.sigma_min)
    return mu, sig


def train_surrogate(
    key: Array,
    Z: Array,
    dX: Array,
    sde_cfg: SDEConfig,
    net: SurrogateNetConfig,
    tr: SurrogateTrainConfig,
) -> Tuple[Dict, Dict[str, Array]]:
    """Learn mu(t,x) and sigma(t,x) from increments."""
    dt = DTYPE(sde_cfg.dt)

    Z_norm, stats = _normalize_z(Z)

    params = init_surrogate_params(key, net)

    opt = optax.chain(
        optax.clip_by_global_norm(tr.clip_norm),
        optax.adamw(tr.lr, weight_decay=tr.weight_decay),
    )
    opt_state = opt.init(params)

    n = Z.shape[0]
    key = jax.random.PRNGKey(tr.seed)

    def mu_sigma_at(params_local, z_norm_batch):
        return surrogate_mu_sigma(params_local, z_norm_batch, net)

    def nll_loss(params_local, z_norm_batch, dX_batch):
        mu, sig = mu_sigma_at(params_local, z_norm_batch)  # (B,1),(B,1)
        # Gaussian conditional likelihood for increments
        mean = mu * dt
        var = (sig * sig) * dt + DTYPE(1e-12)
        resid = dX_batch - mean
        nll = 0.5 * (resid * resid) / var + 0.5 * jnp.log(var)
        return jnp.mean(nll)

    # derivative penalties (encourage smoothness / simplicity)
    def grad_penalties(params_local, z_norm_batch):
        def mu_scalar(z):
            return mu_sigma_at(params_local, z[None, :])[0][0, 0]
        def sig_scalar(z):
            return mu_sigma_at(params_local, z[None, :])[1][0, 0]

        g_mu = jax.vmap(jax.grad(mu_scalar))(z_norm_batch)      # (B,2)
        g_sig = jax.vmap(jax.grad(sig_scalar))(z_norm_batch)    # (B,2)
        return jnp.mean(g_mu * g_mu), jnp.mean(g_sig * g_sig)

    def amp_l1(params_local):
        if net.use_amp_residual_sigma and (params_local["sig_amp"] is not None):
            return jnp.mean(jnp.abs(params_local["sig_amp"]))
        return DTYPE(0.0)

    @jax.jit
    def step(params_local, opt_state_local, key_local):
        key_local, k_idx = jax.random.split(key_local, 2)
        idx = jax.random.randint(k_idx, (tr.batch_size,), 0, n)
        z_b = Z_norm[idx]
        dx_b = dX[idx]

        def total(p):
            base = nll_loss(p, z_b, dx_b)
            gmu, gsig = grad_penalties(p, z_b)
            reg = DTYPE(tr.w_grad_mu) * gmu + DTYPE(tr.w_grad_sigma) * gsig + DTYPE(tr.w_amp_l1) * amp_l1(p)
            return base + reg, (base, gmu, gsig, amp_l1(p))

        (loss, (base, gmu, gsig, a1)), grads = jax.value_and_grad(total, has_aux=True)(params_local)
        updates, opt_state_local = opt.update(grads, opt_state_local, params_local)
        params_local = optax.apply_updates(params_local, updates)

        aux = {"loss": loss, "nll": base, "grad_mu": gmu, "grad_sig": gsig, "amp_l1": a1}
        return params_local, opt_state_local, key_local, aux

    print("\n=== Stage A: learn drift/diffusion from trajectory increments ===")
    for it in range(1, tr.steps + 1):
        params, opt_state, key, aux = step(params, opt_state, key)
        if (it % tr.print_every) == 0 or it == 1:
            print(
                f"[surrogate] step {it:6d}  loss={float(aux['loss']):.3e}  "
                f"nll={float(aux['nll']):.3e}  grad_mu={float(aux['grad_mu']):.2e}  "
                f"grad_sig={float(aux['grad_sig']):.2e}  amp_l1={float(aux['amp_l1']):.2e}"
            )

    return params, stats


# =============================================================================
# 3) Symmetry learning for variable-coefficient FP (Gaeta–Quintero)
# =============================================================================

@dataclass(frozen=True)
class DomainFP:
    t_min: float
    t_max: float
    x_min: float
    x_max: float


@dataclass(frozen=True)
class TrainConfigFP:
    seed: int = 0
    m: int = 6  # how many generators to learn jointly

    # network
    hidden: Tuple[int, ...] = (128, 128, 128)
    activation: str = "swish"  # swish|tanh|relu

    # training
    steps: int = 20000
    batch: int = 512
    lr: float = 3e-4
    clip_norm: float = 1.0

    # curriculum on t_max
    use_curriculum: bool = True
    t_max_start: float = 0.6
    t_max_end: float = 2.0
    curriculum_steps: int = 20000

    # losses
    w_det: float = 1.0
    w_ind: float = 0.8
    w_lie: float = 0.05
    w_col: float = 0.08
    w_logdet: float = 0.05
    ramp_steps: int = 15000

    # additional paper-aligned structure losses (kept small by default)
    w_s2_jacobi: float = 2e-3
    w_s3_skewsym: float = 1e-3
    w_s4_bilinearity: float = 1e-3
    w_s9_after_flow: float = 1e-3

    # keep the original (tau,xi,beta) closure loss as a stabilizer (small weight)
    w_lie_full3: float = 1e-2

    # lie closure
    lie_pairs: int = 12
    ridge: float = 1e-6

    # diversity numerics
    min_col_norm: float = 1e-3
    gram_eps: float = 1e-4

    # logging / eval
    log_every: int = 1000
    eval_every: int = 5000
    eval_grid_t: int = 12
    eval_grid_x: int = 64


def act_fp(x: Array, name: str) -> Array:
    if name == "swish":
        return x * jax.nn.sigmoid(x)
    if name == "tanh":
        return jnp.tanh(x)
    if name == "relu":
        return jax.nn.relu(x)
    raise ValueError(f"Unknown activation: {name}")


def mlp_init_fp(key, in_dim: int, out_dim: int, hidden: Tuple[int, ...]) -> Dict:
    keys = jax.random.split(key, num=len(hidden) + 1)
    dims = (in_dim,) + tuple(hidden) + (out_dim,)
    params = {"W": [], "b": []}
    for i in range(len(dims) - 1):
        k = keys[i]
        fan_in, fan_out = dims[i], dims[i + 1]
        lim = math.sqrt(6.0 / float(fan_in + fan_out))
        W = jax.random.uniform(k, (fan_in, fan_out), minval=-lim, maxval=lim, dtype=DTYPE)
        b = jnp.zeros((fan_out,), dtype=DTYPE)
        params["W"].append(W)
        params["b"].append(b)
    return params


def mlp_apply_fp(params: Dict, x: Array, activation: str) -> Array:
    h = x
    for i in range(len(params["W"]) - 1):
        h = h @ params["W"][i] + params["b"][i]
        h = act_fp(h, activation)
    return h @ params["W"][-1] + params["b"][-1]


def normalize_t(dom: DomainFP, t: Array) -> Array:
    tn = (t - DTYPE(dom.t_min)) / (DTYPE(dom.t_max) - DTYPE(dom.t_min) + DTYPE(1e-12))
    return DTYPE(2.0) * tn - DTYPE(1.0)


def normalize_tx(dom: DomainFP, tx: Array) -> Array:
    t, x = tx[..., 0], tx[..., 1]
    tn = (t - DTYPE(dom.t_min)) / (DTYPE(dom.t_max) - DTYPE(dom.t_min) + DTYPE(1e-12))
    xn = (x - DTYPE(dom.x_min)) / (DTYPE(dom.x_max) - DTYPE(dom.x_min) + DTYPE(1e-12))
    return jnp.stack([DTYPE(2.0) * tn - DTYPE(1.0), DTYPE(2.0) * xn - DTYPE(1.0)], axis=-1)


def sample_batch_fp(key, dom: DomainFP, batch: int, t_max: Array) -> Array:
    k1, k2 = jax.random.split(key, 2)
    t_hi = jnp.maximum(DTYPE(dom.t_min) + DTYPE(1e-6), t_max)
    t = jax.random.uniform(k1, (batch,), minval=DTYPE(dom.t_min), maxval=t_hi, dtype=DTYPE)
    x = jax.random.uniform(k2, (batch,), minval=DTYPE(dom.x_min), maxval=DTYPE(dom.x_max), dtype=DTYPE)
    return jnp.stack([t, x], axis=-1)


def orthonormal_basis(M: Array, eps: float = 1e-8) -> Array:
    Q, R = jnp.linalg.qr(M, mode="reduced")
    diag = jnp.abs(jnp.diag(R))
    mx = jnp.max(diag + eps)
    keep = diag > eps * mx
    keep = jnp.where(
        jnp.any(keep),
        keep,
        jnp.concatenate([jnp.array([True]), jnp.zeros((keep.shape[0] - 1,), dtype=bool)]),
    )
    return Q[:, keep]


def principal_angles(V: Array, W: Array) -> Array:
    Qv = orthonormal_basis(V)
    Qw = orthonormal_basis(W)
    s = jnp.linalg.svd(Qv.T @ Qw, compute_uv=False)
    s = jnp.clip(s, 0.0, 1.0)
    return jnp.sort(jnp.arccos(s))


def best_mixing_residual(V: Array, W: Array) -> float:
    WT_W = W.T @ W + DTYPE(1e-8) * jnp.eye(W.shape[1], dtype=DTYPE)
    A = jnp.linalg.solve(WT_W, W.T @ V)
    resid = jnp.linalg.norm(V - W @ A) / (jnp.linalg.norm(V) + DTYPE(1e-12))
    return float(resid)


def init_params_fp(key, cfg: TrainConfigFP) -> Dict:
    k1, k2, k3 = jax.random.split(key, 3)
    return {
        "tau": mlp_init_fp(k1, in_dim=1, out_dim=cfg.m, hidden=cfg.hidden),
        "xi": mlp_init_fp(k2, in_dim=2, out_dim=1 * cfg.m, hidden=cfg.hidden),
        "beta": mlp_init_fp(k3, in_dim=2, out_dim=cfg.m, hidden=cfg.hidden),
    }


def forward_fp(params: Dict, dom: DomainFP, cfg: TrainConfigFP, tx: Array):
    t = tx[..., 0]
    tn = normalize_t(dom, t)[..., None]
    z = normalize_tx(dom, tx)
    tau = mlp_apply_fp(params["tau"], tn, cfg.activation)  # (...,m)
    xi_flat = mlp_apply_fp(params["xi"], z, cfg.activation)
    xi = xi_flat.reshape(*xi_flat.shape[:-1], cfg.m, 1)  # (...,m,1)
    beta = mlp_apply_fp(params["beta"], z, cfg.activation)  # (...,m)
    return tau, xi, beta


def t_max_curriculum_jax(cfg: TrainConfigFP, dom: DomainFP, step: Array) -> Array:
    if not cfg.use_curriculum:
        return DTYPE(dom.t_max)
    s = jnp.minimum(DTYPE(step), DTYPE(cfg.curriculum_steps))
    frac = s / DTYPE(max(cfg.curriculum_steps, 1))
    return DTYPE(cfg.t_max_start) + frac * DTYPE(cfg.t_max_end - cfg.t_max_start)


def make_losses_varcoeff(
    dom: DomainFP,
    cfg: TrainConfigFP,
    mu_fn: Callable[[Array, Array], Array],
    sig_fn: Callable[[Array, Array], Array],
):
    """
    Symmetry-learning loss suite.

    We keep the original working losses (FP determining equations S8, column-independence,
    logdet barrier, column-norm regularizer, and the original 3-component Lie-closure loss),
    but we *re-factor* them to align with the paper loss naming scheme from `loss functions.py`:

      - S8: FP determining equations (Gaeta–Quintero Eq. 4.12 specialized to 1D)
      - S5: column independence (Gram ~ I)
      - S1: Lie bracket closure + constancy of structure coefficients on (t,x) fields (tau,xi)
      - S2: Jacobi identity on learned structure coefficients (built from S1 projection)
      - S3: skew-symmetry of structure coefficients
      - S4: (numerical) linearity check of the S1 projection coefficients
      - S9: "after-flow" robustness: evaluate S8 at points flowed a small epsilon along a few generators
    """

    # ---------------------------- FP coefficients (1D) ----------------------------
    def sigma2(tt, xx):
        s = sig_fn(tt, xx)
        return s * s

    def A(tt, xx):
        # Eq (4.2): A = -1/2 (σ σ^T) ; in 1D this is -1/2 σ^2
        return -DTYPE(0.5) * sigma2(tt, xx)

    def B(tt, xx):
        # Eq (4.2): B = f - ∂_x(σ σ^T) ; in 1D: f - (σ^2)_x
        s2_x = jax.grad(lambda x_: sigma2(tt, x_))(xx)
        return mu_fn(tt, xx) - s2_x

    def C(tt, xx):
        # Eq (4.2): C = (∂_x f) - 1/2 ∂_{xx}(σ σ^T) ; in 1D: f_x - 1/2 (σ^2)_{xx}
        f_x = jax.grad(lambda x_: mu_fn(tt, x_))(xx)
        s2_xx = jax.grad(lambda x_: jax.grad(lambda x2: sigma2(tt, x2))(x_))(xx)
        return f_x - DTYPE(0.5) * s2_xx

    def coeffs_and_derivs(t_scalar, x_scalar):
        A0 = A(t_scalar, x_scalar)
        B0 = B(t_scalar, x_scalar)
        C0 = C(t_scalar, x_scalar)

        A_t = jax.grad(lambda tt: A(tt, x_scalar))(t_scalar)
        A_x = jax.grad(lambda xx: A(t_scalar, xx))(x_scalar)

        B_t = jax.grad(lambda tt: B(tt, x_scalar))(t_scalar)
        B_x = jax.grad(lambda xx: B(t_scalar, xx))(x_scalar)

        C_t = jax.grad(lambda tt: C(tt, x_scalar))(t_scalar)
        C_x = jax.grad(lambda xx: C(t_scalar, xx))(x_scalar)

        return A0, A_t, A_x, B0, B_t, B_x, C0, C_t, C_x

    # ---------------------------- generator vector eval ----------------------------
    def tau_vec(params, t_scalar):
        t_arr = jnp.asarray([[t_scalar]], dtype=DTYPE)
        tn = normalize_t(dom, t_arr)
        return mlp_apply_fp(params["tau"], tn, cfg.activation)[0]  # (m,)

    def xi_vec(params, z):
        _, xi, _ = forward_fp(params, dom, cfg, z[None, :])
        return xi[0, :, 0]  # (m,)

    def beta_vec(params, z):
        _, _, beta = forward_fp(params, dom, cfg, z[None, :])
        return beta[0]  # (m,)

    def point_derivs(params, z):
        t = z[0]
        tau, xi, beta = forward_fp(params, dom, cfg, z[None, :])
        tau, xi, beta = tau[0], xi[0, :, 0], beta[0]

        tau_t = jax.jacfwd(lambda tt: tau_vec(params, tt))(t)  # (m,)

        J_xi = jax.jacrev(lambda zz: xi_vec(params, zz))(z)  # (m,2)
        xi_t = J_xi[:, 0]
        xi_x = J_xi[:, 1]

        H_xi = jax.jacrev(jax.jacrev(lambda zzz: xi_vec(params, zzz)))(z)  # (m,2,2)
        xi_xx = H_xi[:, 1, 1]

        J_b = jax.jacrev(lambda zz: beta_vec(params, zz))(z)  # (m,2)
        beta_t = J_b[:, 0]
        beta_x = J_b[:, 1]

        H_b = jax.jacrev(jax.jacrev(lambda zzz: beta_vec(params, zzz)))(z)  # (m,2,2)
        beta_xx = H_b[:, 1, 1]

        return tau, tau_t, xi, xi_t, xi_x, xi_xx, beta, beta_t, beta_x, beta_xx

    # ---------------------------- S8: determining equations ----------------------------
    def det_residuals(params, batch):
        def one(z):
            tau, tau_t, xi, xi_t, xi_x, xi_xx, beta, beta_t, beta_x, beta_xx = point_derivs(params, z)
            t_scalar, x_scalar = z[0], z[1]
            A0, A_t, A_x, B0, B_t, B_x, C0, C_t, C_x = coeffs_and_derivs(t_scalar, x_scalar)

            r1 = (tau_t * A0 + tau * A_t) + xi * A_x - DTYPE(2.0) * A0 * xi_x
            r2 = (tau_t * B0 + tau * B_t) - (xi_t - B0 * xi_x + xi * B_x) + DTYPE(2.0) * A0 * beta_x - A0 * xi_xx
            r3 = (tau_t * C0 + tau * C_t) + beta_t + A0 * beta_xx + B0 * beta_x + xi * C_x
            return r1, r2, r3

        r1, r2, r3 = jax.vmap(one)(batch)  # each (B,m)
        return r1, r2, r3

    def s8_fp_determining_loss_1d(params, batch):
        r1, r2, r3 = det_residuals(params, batch)
        loss = jnp.mean(r1 * r1 + r2 * r2 + r3 * r3)
        aux = {
            "s8_r1": jnp.mean(r1 * r1),
            "s8_r2": jnp.mean(r2 * r2),
            "s8_r3": jnp.mean(r3 * r3),
        }
        return loss, aux

    def det_per_gen(params, batch):
        r1, r2, r3 = det_residuals(params, batch)
        per = jnp.mean(r1 * r1 + r2 * r2 + r3 * r3, axis=0)  # (m,)
        return per

    # ---------------------------- shared column stacking ----------------------------
    def stack_cols(params, batch):
        tau, xi, beta = forward_fp(params, dom, cfg, batch)
        Vb = jnp.stack([tau, xi[:, :, 0], beta], axis=-1)  # (B,m,3)
        return jnp.reshape(jnp.transpose(Vb, (0, 2, 1)), (batch.shape[0] * 3, cfg.m))

    # ---------------------------- S5: column independence ----------------------------
    def s5_column_independence_loss(params, batch):
        V = stack_cols(params, batch)  # (3B,m)
        col_norms = jnp.sqrt(jnp.sum(V * V, axis=0) + DTYPE(1e-12))
        Vn = V / col_norms[None, :]
        G = Vn.T @ Vn / DTYPE(Vn.shape[0])
        loss = jnp.mean((G - jnp.eye(cfg.m, dtype=DTYPE)) ** 2)
        aux = {"s5_ind_mse": loss, "min_col": jnp.min(col_norms), "max_col": jnp.max(col_norms)}
        return loss, aux

    # ---------------------------- original stabilizers ----------------------------
    def orig_logdet_loss(params, batch):
        V = stack_cols(params, batch)
        col_norms = jnp.sqrt(jnp.sum(V * V, axis=0) + DTYPE(1e-12))
        Vn = V / col_norms[None, :]
        G = Vn.T @ Vn / DTYPE(Vn.shape[0])
        G = G + DTYPE(cfg.gram_eps) * jnp.eye(cfg.m, dtype=DTYPE)
        sign, ld = jnp.linalg.slogdet(G)
        return -ld  # encourage large det

    def orig_col_norm_loss(params, batch, target: float = 1.0):
        V = stack_cols(params, batch)
        col_norms = jnp.sqrt(jnp.sum(V * V, axis=0) + DTYPE(1e-12))
        tgt = DTYPE(target)
        penalty = jnp.mean((col_norms - tgt) ** 2) + jnp.mean(jax.nn.relu(DTYPE(cfg.min_col_norm) - col_norms) ** 2)
        return penalty

    # Original Lie closure (on (tau,xi,beta) stacked columns) as an extra stabilizer.
    def orig_lie_closure_full3(params, batch, key_lie):
        V = stack_cols(params, batch)  # (3B,m)
        G = V.T @ V + DTYPE(cfg.ridge) * jnp.eye(cfg.m, dtype=DTYPE)

        def eval_fields(z):
            t = z[0]
            return tau_vec(params, t), xi_vec(params, z), beta_vec(params, z)

        def bracket_at_point(z, i, j):
            (tau_i, xi_i, beta_i) = eval_fields(z)
            (tau_j, xi_j, beta_j) = eval_fields(z)

            ti, xi_i, bi = tau_i[i], xi_i[i], beta_i[i]
            tj, xj, bj = tau_j[j], xi_j[j], beta_j[j]

            def tj_fun(t):
                return tau_vec(params, t)[j]
            def xj_fun(z_):
                return xi_vec(params, z_)[j]
            def bj_fun(z_):
                return beta_vec(params, z_)[j]

            dt_tj = jax.grad(tj_fun)(z[0])
            grad_xj = jax.grad(xj_fun)(z)         # (2,)
            grad_bj = jax.grad(bj_fun)(z)         # (2,)

            def ti_fun(t):
                return tau_vec(params, t)[i]
            def xi_fun(z_):
                return xi_vec(params, z_)[i]
            def bi_fun(z_):
                return beta_vec(params, z_)[i]

            dt_ti = jax.grad(ti_fun)(z[0])
            grad_xi = jax.grad(xi_fun)(z)         # (2,)
            grad_bi = jax.grad(bi_fun)(z)         # (2,)

            bt = ti * dt_tj - tj * dt_ti
            bx = ti * grad_xj[0] + xi_i * grad_xj[1] - (tj * grad_xi[0] + xj * grad_xi[1])
            bb = ti * grad_bj[0] + xi_i * grad_bj[1] - (tj * grad_bi[0] + xj * grad_bi[1])
            return bt, bx, bb

        k1, k2 = jax.random.split(key_lie, 2)
        ii = jax.random.randint(k1, (cfg.lie_pairs,), 0, cfg.m)
        jj = jax.random.randint(k2, (cfg.lie_pairs,), 0, cfg.m)

        def one_pair(pair_idx):
            i = ii[pair_idx]
            j = jj[pair_idx]

            def one_point(z):
                bt, bx, bb = bracket_at_point(z, i, j)
                return jnp.stack([bt, bx, bb], axis=0)  # (3,)

            Bvec = jax.vmap(one_point)(batch)  # (B,3)
            v = jnp.reshape(Bvec, (batch.shape[0] * 3,))  # (3B,)

            rhs = V.T @ v
            c = jnp.linalg.solve(G, rhs)
            r = v - V @ c
            rel = jnp.mean(r * r) / (jnp.mean(v * v) + DTYPE(1e-12))
            return rel

        rels = jax.vmap(one_pair)(jnp.arange(cfg.lie_pairs))
        loss = jnp.mean(rels)
        return loss, {"orig_lie_rel_mse": loss}

    # ---------------------------- S1–S4: Lie algebra structure on (tau,xi) ----------------------------
    # Ordered pairs (i,j) with i != j
    idx = jnp.arange(cfg.m, dtype=jnp.int32)
    idx_i = jnp.repeat(idx, repeats=cfg.m - 1)
    base = jnp.arange(cfg.m - 1, dtype=jnp.int32)
    i_col = idx[:, None]
    idx_j = (base + (base >= i_col).astype(jnp.int32)).reshape(-1)
    K = int(idx_i.shape[0])

    reg_s1 = DTYPE(cfg.ridge)

    # Triples i<j<k for Jacobi (computed once on host, then baked in)
    triples = [(i, j, k) for i in range(cfg.m) for j in range(i + 1, cfg.m) for k in range(j + 1, cfg.m)]
    tri_i = jnp.asarray([t[0] for t in triples], dtype=jnp.int32) if triples else jnp.zeros((0,), dtype=jnp.int32)
    tri_j = jnp.asarray([t[1] for t in triples], dtype=jnp.int32) if triples else jnp.zeros((0,), dtype=jnp.int32)
    tri_k = jnp.asarray([t[2] for t in triples], dtype=jnp.int32) if triples else jnp.zeros((0,), dtype=jnp.int32)

    # Pair indices for S4 linearity check (projection is linear, so this should be ~0)
    P_lin = min(8, max(K // 2, 1))
    p_lin = jnp.arange(P_lin, dtype=jnp.int32)
    q_lin = (p_lin + 1) % K

    def _s1_point_err_and_C(tau, xi, tau_t, xi_t, xi_x):
        # V: (2,m)
        V = jnp.stack([tau, xi], axis=0)  # (2,m)

        tau_i, tau_j = tau[idx_i], tau[idx_j]  # (K,)
        xi_i, xi_j = xi[idx_i], xi[idx_j]
        tau_t_i, tau_t_j = tau_t[idx_i], tau_t[idx_j]
        xi_t_i, xi_t_j = xi_t[idx_i], xi_t[idx_j]
        xi_x_i, xi_x_j = xi_x[idx_i], xi_x[idx_j]

        a = tau_i * tau_t_j - tau_j * tau_t_i
        b = tau_i * xi_t_j + xi_i * xi_x_j - tau_j * xi_t_i - xi_j * xi_x_i
        Bmat = jnp.stack([a, b], axis=0)  # (2,K)

        G = V @ V.T
        G_reg = G + reg_s1 * jnp.eye(2, dtype=DTYPE)
        X = jnp.linalg.solve(G_reg, Bmat)      # (2,K)
        Cmat = V.T @ X                         # (m,K)
        PB = V @ Cmat                          # (2,K)
        E = Bmat - PB

        closure_mse = jnp.mean(E * E)
        return closure_mse, Cmat, V, Bmat, G_reg

    def s1_lie_loss(params, tx_batch: jnp.ndarray):
        def eval_at_z(z):
            tau, tau_t, xi, xi_t, xi_x, _, _, _, _, _ = point_derivs(params, z)
            return tau, xi, tau_t, xi_t, xi_x

        taus, xis, tau_ts, xi_ts, xi_xs = jax.vmap(eval_at_z)(tx_batch)  # each (B,m)

        def one_point(tau, xi, tau_t, xi_t, xi_x):
            return _s1_point_err_and_C(tau, xi, tau_t, xi_t, xi_x)[:2]

        closure_mse_pts, Cs = jax.vmap(one_point)(taus, xis, tau_ts, xi_ts, xi_xs)
        closure_mse = jnp.mean(closure_mse_pts)
        C_var = jnp.mean(jnp.var(Cs, axis=0))  # scalar

        total = closure_mse + C_var
        aux = {
            "s1_closure_mse": closure_mse,
            "s1_C_var": C_var,
        }
        return total, aux

    def _Cbar_to_c_tensor(Cbar):
        # Cbar: (m,K) -> c: (m,m,m) with c[i,j,k]
        def one_k(k):
            M = jnp.zeros((cfg.m, cfg.m), dtype=DTYPE)
            return M.at[idx_i, idx_j].set(Cbar[k, :])
        mats = jax.vmap(one_k)(jnp.arange(cfg.m, dtype=jnp.int32))  # (m,m,m) but indexed [k,i,j]
        c = jnp.transpose(mats, (1, 2, 0))  # (i,j,k)
        return c

    def s2_jacobi_loss(params, tx_batch: jnp.ndarray):
        # Build structure constants from batch-averaged coefficients from S1.
        def eval_at_z(z):
            tau, tau_t, xi, xi_t, xi_x, _, _, _, _, _ = point_derivs(params, z)
            _, Cmat, _, _, _ = _s1_point_err_and_C(tau, xi, tau_t, xi_t, xi_x)
            return Cmat

        Cs = jax.vmap(eval_at_z)(tx_batch)  # (B,m,K)
        Cbar = jnp.mean(Cs, axis=0)         # (m,K)
        c = _Cbar_to_c_tensor(Cbar)         # (m,m,m)

        if tri_i.shape[0] == 0:
            return DTYPE(0.0), {"s2_jacobi_mse": DTYPE(0.0)}

        def jac_one(i, j, k):
            # vector over m
            term1 = c[i, j, :] @ c[:, k, :]  # (m,)
            term2 = c[j, k, :] @ c[:, i, :]
            term3 = c[k, i, :] @ c[:, j, :]
            jac = term1 + term2 + term3
            return jnp.mean(jac * jac)

        vals = jax.vmap(jac_one)(tri_i, tri_j, tri_k)
        loss = jnp.mean(vals)
        return loss, {"s2_jacobi_mse": loss}

    def s3_skewsym_loss(params, tx_batch: jnp.ndarray):
        # Use same averaged c tensor as S2.
        def eval_at_z(z):
            tau, tau_t, xi, xi_t, xi_x, _, _, _, _, _ = point_derivs(params, z)
            _, Cmat, _, _, _ = _s1_point_err_and_C(tau, xi, tau_t, xi_t, xi_x)
            return Cmat

        Cs = jax.vmap(eval_at_z)(tx_batch)  # (B,m,K)
        Cbar = jnp.mean(Cs, axis=0)
        c = _Cbar_to_c_tensor(Cbar)  # (m,m,m)

        skew = c + jnp.swapaxes(c, 0, 1)  # c[i,j,k] + c[j,i,k]
        skew_mse = jnp.mean(skew * skew)

        diag = c[jnp.arange(cfg.m), jnp.arange(cfg.m), :]  # (m,m)
        diag_mse = jnp.mean(diag * diag)

        loss = skew_mse + diag_mse
        return loss, {"s3_skew_mse": skew_mse, "s3_diag_mse": diag_mse}

    def s4_bilinearity_loss(params, tx_batch: jnp.ndarray):
        # Numerical check: projection coefficients are linear in the bracket argument B.
        def eval_at_z(z):
            tau, tau_t, xi, xi_t, xi_x, _, _, _, _, _ = point_derivs(params, z)
            closure_mse, Cmat, V, Bmat, G_reg = _s1_point_err_and_C(tau, xi, tau_t, xi_t, xi_x)

            def one_pair(p, q):
                Bsum = Bmat[:, p] + Bmat[:, q]  # (2,)
                Xsum = jnp.linalg.solve(G_reg, Bsum)  # (2,)
                Csum = V.T @ Xsum                     # (m,)
                return jnp.mean((Csum - (Cmat[:, p] + Cmat[:, q])) ** 2)

            errs = jax.vmap(one_pair)(p_lin, q_lin)
            return jnp.mean(errs)

        per_pt = jax.vmap(eval_at_z)(tx_batch)
        loss = jnp.mean(per_pt)
        return loss, {"s4_lin_mse": loss}

    # ---------------------------- S9: after-flow robustness ----------------------------
    def s9_fp_after_flow_loss_1d(params, tx_batch: jnp.ndarray, eps: float = 1e-2):
        # Flow a small step along a couple generators and evaluate S8 there as a robustness check.
        tau, xi, _ = forward_fp(params, dom, cfg, tx_batch)  # (B,m),(B,m,1)
        n_use = min(2, cfg.m)

        def flow_i(i):
            dt = eps * tau[:, i]
            dx = eps * xi[:, i, 0]
            t2 = jnp.clip(tx_batch[:, 0] + dt, DTYPE(dom.t_min), DTYPE(dom.t_max))
            x2 = jnp.clip(tx_batch[:, 1] + dx, DTYPE(dom.x_min), DTYPE(dom.x_max))
            tx2 = jnp.stack([t2, x2], axis=1)
            d2, _ = s8_fp_determining_loss_1d(params, tx2)
            return d2

        vals = jax.vmap(flow_i)(jnp.arange(n_use, dtype=jnp.int32))
        loss = jnp.mean(vals)
        return loss, {"s9_det_flow": loss}

    return (
        jax.jit(s8_fp_determining_loss_1d),
        jax.jit(det_per_gen),
        jax.jit(s5_column_independence_loss),
        jax.jit(orig_logdet_loss),
        jax.jit(orig_col_norm_loss),
        jax.jit(s1_lie_loss),
        jax.jit(s2_jacobi_loss),
        jax.jit(s3_skewsym_loss),
        jax.jit(s4_bilinearity_loss),
        jax.jit(s9_fp_after_flow_loss_1d),
        jax.jit(orig_lie_closure_full3),
        jax.jit(stack_cols),
    )
# =============================================================================
# (Optional) ground truth evaluation for the Brownian-motion demo
# =============================================================================

def ground_truth_heat1d(tx: Array, sigma0: float):
    # Basis for the constant-diffusion heat case:
    t = tx[:, 0]
    x = tx[:, 1]
    N = tx.shape[0]
    tau = jnp.zeros((N, 6), dtype=DTYPE)
    xi = jnp.zeros((N, 6, 1), dtype=DTYPE)
    beta = jnp.zeros((N, 6), dtype=DTYPE)

    s0 = DTYPE(sigma0)
    s02 = s0 * s0

    tau = tau.at[:, 0].set(DTYPE(1.0))              # v1: ∂t
    xi = xi.at[:, 1, 0].set(DTYPE(1.0))             # v2: ∂x
    beta = beta.at[:, 2].set(DTYPE(1.0))            # v3: u∂u

    xi = xi.at[:, 3, 0].set(s02 * t)                # v4
    beta = beta.at[:, 3].set(-s0 * x)

    tau = tau.at[:, 4].set(DTYPE(2.0) * t)          # v5
    xi = xi.at[:, 4, 0].set(x)

    tau = tau.at[:, 5].set(t * t)                   # v6
    xi = xi.at[:, 5, 0].set(x * t)
    beta = beta.at[:, 5].set(-DTYPE(0.5) * (t + (x * x) / (s02 + DTYPE(1e-12))))

    return tau, xi, beta


def stack_cols_gt(tx: Array, sigma0: float) -> Array:
    tau, xi, beta = ground_truth_heat1d(tx, sigma0)
    Vb = jnp.stack([tau, xi[:, :, 0], beta], axis=-1)  # (N,6,3)
    return jnp.reshape(jnp.transpose(Vb, (0, 2, 1)), (tx.shape[0] * 3, 6))


def eval_grid(dom: DomainFP, cfg: TrainConfigFP) -> Array:
    t = jnp.linspace(DTYPE(dom.t_min), DTYPE(dom.t_max), cfg.eval_grid_t, dtype=DTYPE)
    x = jnp.linspace(DTYPE(dom.x_min), DTYPE(dom.x_max), cfg.eval_grid_x, dtype=DTYPE)
    T, X = jnp.meshgrid(t, x, indexing="ij")
    return jnp.stack([T.reshape(-1), X.reshape(-1)], axis=1)


# =============================================================================
# main
# =============================================================================

def main():
    # -------------------- configs --------------------
    sde_cfg = SDEConfig()
    net_cfg = SurrogateNetConfig()
    tr_cfg = SurrogateTrainConfig()

    fp_cfg = TrainConfigFP()

    print("=== SDE -> learned FP -> FP symmetries (variable coefficients) ===")
    print(f"JAX devices: {jax.devices()}")
    print(f"SDE: T={sde_cfg.T} dt={sde_cfg.dt} n_traj={sde_cfg.n_traj}")
    print(f"Surrogate: hidden={net_cfg.hidden} depth={net_cfg.depth} act={net_cfg.activation}")
    print(f"Symmetry: m={fp_cfg.m} hidden={fp_cfg.hidden} act={fp_cfg.activation}")
    print()

    # -------------------- simulate data --------------------
    key = jax.random.PRNGKey(0)
    TX_all, Z, dX = simulate_traj_and_increments(key, sde_cfg)

    # set FP domain from data (with margin)
    t_min = 1e-3
    t_max = float(sde_cfg.T)
    x_vals = np.asarray(TX_all[:, 1])
    q_lo, q_hi = np.quantile(x_vals, [0.001, 0.999])
    x_pad = 0.25 * (q_hi - q_lo + 1e-6)
    dom = DomainFP(t_min=t_min, t_max=t_max, x_min=float(q_lo - x_pad), x_max=float(q_hi + x_pad))

    print(f"Data-informed FP domain: t∈[{dom.t_min:.3g},{dom.t_max:.3g}] x∈[{dom.x_min:.3g},{dom.x_max:.3g}]")
    print()

    # -------------------- learn mu,sigma --------------------
    k_sur = jax.random.PRNGKey(tr_cfg.seed)
    params_surr, stats = train_surrogate(k_sur, Z, dX, sde_cfg, net_cfg, tr_cfg)

    # freeze learned mu,sigma as scalar-callable functions for the FP loss
    def mu_fn(t_scalar: Array, x_scalar: Array) -> Array:
        z = jnp.stack([t_scalar, x_scalar], axis=0)[None, :]  # (1,2)
        z_norm = (z - stats["mean"][None, :]) / stats["std"][None, :]
        mu, _ = surrogate_mu_sigma(params_surr, z_norm, net_cfg)
        return mu[0, 0]

    def sig_fn(t_scalar: Array, x_scalar: Array) -> Array:
        z = jnp.stack([t_scalar, x_scalar], axis=0)[None, :]  # (1,2)
        z_norm = (z - stats["mean"][None, :]) / stats["std"][None, :]
        _, sig = surrogate_mu_sigma(params_surr, z_norm, net_cfg)
        return sig[0, 0]


    tx_probe = jnp.array([[0.25, 0.0], [1.0, 0.5], [1.5, -0.5]], dtype=DTYPE)
    mu_probe = jax.vmap(lambda z: mu_fn(z[0], z[1]))(tx_probe)
    sig_probe = jax.vmap(lambda z: sig_fn(z[0], z[1]))(tx_probe)
    print("[surrogate] probe mu(t,x):", np.asarray(mu_probe))
    print("[surrogate] probe sigma(t,x):", np.asarray(sig_probe))
    print()

    # -------------------- learn FP symmetries --------------------
    print("=== Stage B: learn FP symmetries from learned (mu,sigma) ===")
    key = jax.random.PRNGKey(fp_cfg.seed)
    params = init_params_fp(key, fp_cfg)

    s8_loss_fn, det_per_gen_fn, s5_loss_fn, logdet_loss_fn, col_loss_fn, s1_loss_fn, s2_loss_fn, s3_loss_fn, s4_loss_fn, s9_loss_fn, lie_full3_loss_fn, stack_cols_fn = \
        make_losses_varcoeff(dom, fp_cfg, mu_fn=mu_fn, sig_fn=sig_fn)

    opt = optax.chain(
        optax.clip_by_global_norm(fp_cfg.clip_norm),
        optax.adam(fp_cfg.lr),
    )
    opt_state = opt.init(params)

    # evaluation grid + GT span (only meaningful for the demo constant-diffusion SDE)
    pts_eval = eval_grid(dom, fp_cfg)
    W_eval = stack_cols_gt(pts_eval, sde_cfg.sigma0)

    best_params = params
    best_score = float("inf")
    best_step = 0

    @jax.jit
    def step_fn(params, opt_state, key, step):
        t_max_cur = t_max_curriculum_jax(fp_cfg, dom, step)
        key, k_batch, k_lie = jax.random.split(key, 3)
        batch = sample_batch_fp(k_batch, dom, fp_cfg.batch, t_max_cur)

        ramp = jnp.minimum(DTYPE(1.0), DTYPE(step) / DTYPE(max(fp_cfg.ramp_steps, 1)))
        w_ind = DTYPE(fp_cfg.w_ind) * ramp
        w_lie = DTYPE(fp_cfg.w_lie) * ramp
        w_logdet = DTYPE(fp_cfg.w_logdet) * ramp


        def total_loss(p):
            s8, s8aux = s8_loss_fn(p, batch)
            s5, s5aux = s5_loss_fn(p, batch)
            s1, s1aux = s1_loss_fn(p, batch)
            s2, s2aux = s2_loss_fn(p, batch)
            s3, s3aux = s3_loss_fn(p, batch)
            s4, s4aux = s4_loss_fn(p, batch)
            s9, s9aux = s9_loss_fn(p, batch)
            ld = logdet_loss_fn(p, batch)
            col = col_loss_fn(p, batch, target=1.0)
            lie3, lie3aux = lie_full3_loss_fn(p, batch, k_lie)

            w_s2 = DTYPE(fp_cfg.w_s2_jacobi) * ramp
            w_s3 = DTYPE(fp_cfg.w_s3_skewsym) * ramp
            w_s4 = DTYPE(fp_cfg.w_s4_bilinearity) * ramp
            w_s9 = DTYPE(fp_cfg.w_s9_after_flow) * ramp
            w_lie3 = DTYPE(fp_cfg.w_lie_full3) * ramp

            tot = (
                DTYPE(fp_cfg.w_det) * s8
                + w_ind * s5
                + w_logdet * ld
                + w_lie * s1
                + DTYPE(fp_cfg.w_col) * col
                + w_s2 * s2
                + w_s3 * s3
                + w_s4 * s4
                + w_s9 * s9
                + w_lie3 * lie3
            )

            aux = {
                "tot": tot,
                # paper-aligned terms
                "s8": s8,
                "s8_r1": s8aux["s8_r1"],
                "s8_r2": s8aux["s8_r2"],
                "s8_r3": s8aux["s8_r3"],
                "s5": s5,
                "s5_ind_mse": s5aux["s5_ind_mse"],
                "s1": s1,
                "s1_closure_mse": s1aux["s1_closure_mse"],
                "s1_C_var": s1aux["s1_C_var"],
                "s2": s2,
                "s2_jacobi_mse": s2aux["s2_jacobi_mse"],
                "s3": s3,
                "s3_skew_mse": s3aux["s3_skew_mse"],
                "s3_diag_mse": s3aux["s3_diag_mse"],
                "s4": s4,
                "s4_lin_mse": s4aux["s4_lin_mse"],
                "s9": s9,
                "s9_det_flow": s9aux["s9_det_flow"],
                # original stabilizers
                "logdet": ld,
                "col": col,
                "orig_lie3": lie3,
                "orig_lie_rel_mse": lie3aux["orig_lie_rel_mse"],
                # misc
                "min_col": s5aux["min_col"],
                "max_col": s5aux["max_col"],
                "t_max": t_max_cur,
                "ramp": ramp,
                "w_s2": w_s2,
                "w_s3": w_s3,
                "w_s4": w_s4,
                "w_s9": w_s9,
                "w_lie3": w_lie3,
            }
            return tot, aux

        (loss, aux), grads = jax.value_and_grad(total_loss, has_aux=True)(params)
        updates, opt_state = opt.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, key, aux

    def eval_angles(tag: str, params_eval):
        V_eval = stack_cols_fn(params_eval, pts_eval)
        resid = best_mixing_residual(V_eval, W_eval)
        ang = principal_angles(W_eval, V_eval)
        ang_np = np.asarray(ang)
        print(f"[eval:{tag}]  resid ||V-WA||/||V|| = {resid:.3e}")
        for k, a in enumerate(ang_np, start=1):
            print(f"  angle {k:2d}: {a:.6f} rad  = {a*180.0/math.pi:.4f} deg")

    for step in range(1, fp_cfg.steps + 1):
        params, opt_state, key, aux = step_fn(params, opt_state, key, jnp.asarray(step, dtype=DTYPE))

        # combined score
        score = float(aux["s8"] + DTYPE(0.25) * aux["s5"] + DTYPE(0.25) * aux["col"] + DTYPE(0.1) * aux["s1"])
        if score < best_score:
            best_score = score
            best_params = params
            best_step = step

        if (step % fp_cfg.log_every) == 0 or step == 1:

            print(
                f"[train] step {step:6d}  tot={float(aux['tot']):.3e}  "
                f"S8={float(aux['s8']):.3e} (r1={float(aux['s8_r1']):.2e}, r2={float(aux['s8_r2']):.2e}, r3={float(aux['s8_r3']):.2e})  "
                f"S5={float(aux['s5']):.3e}  "
                f"S1={float(aux['s1']):.3e} (cl={float(aux['s1_closure_mse']):.2e}, varC={float(aux['s1_C_var']):.2e})  "
                f"S2={float(aux['s2']):.2e}  S3={float(aux['s3']):.2e}  S4={float(aux['s4']):.2e}  S9={float(aux['s9']):.2e}  "
                f"logdet={float(aux['logdet']):.3e}  col={float(aux['col']):.3e}  origLie3={float(aux['orig_lie3']):.3e}  "
                f"min/max_col={float(aux['min_col']):.2e}/{float(aux['max_col']):.2e}  "
                f"t_max={float(aux['t_max']):.3g} ramp={float(aux['ramp']):.2f}  "
                f"(w_s2={float(aux['w_s2']):.1e}, w_s3={float(aux['w_s3']):.1e}, w_s4={float(aux['w_s4']):.1e}, w_s9={float(aux['w_s9']):.1e}, w_lie3={float(aux['w_lie3']):.1e})"
            )

        if (step % fp_cfg.eval_every) == 0:
            eval_angles("current", params)
            eval_angles(f"best_score@{best_step}", best_params)

            # per-generator det residuals (help diagnose degeneracy)
            batch_eval = sample_batch_fp(jax.random.PRNGKey(1234), dom, fp_cfg.batch, DTYPE(dom.t_max))
            per = np.asarray(det_per_gen_fn(params, batch_eval))
            print("  per-gen det mse:", np.array2string(per, precision=3, floatmode="fixed"))

    print("\n=== Final evaluation ===")
    eval_angles("current", params)
    eval_angles(f"best_score@{best_step}", best_params)


if __name__ == "__main__":
    main()